----------------------------------------------------------------------------------------------------

# ***Gerenciamento de Transações - SQLite***

A DB API também nos permite gerenciar transações, o que é crucial para manter a integridade dos dados.

----------------------------------------------------------------------------------------------------

# ***1. O que é uma Transação?*** 

Uma transação é um conjunto de operações que devem ser executadas como uma unidade. Ou tudo é executado com sucesso, ou nada é aplicado. Isso garante a integridade dos dados.

----------------------------------------------------------------------------------------------------

## ***1.1. O conceito se baseia nas propriedades ACID:*** 

| Propriedade       | Significado                 |
| ----------- | -------------------------- |
| Atomicity    | Tudo ou nada — operações parciais não existem |
| Consistency  | O banco sempre vai de um estado válido para outro|
| Isolation      | Transações simultâneas não interferem entre si|
| Durability   | Após o `commit`, os dados são permanentes  |

----------------------------------------------------------------------------------------------------

## ***1.2. Controle Manual com try/except*** 

A forma mais robusta e recomendada:

Para estudo, usaremos os códigos a seguir de exemplo no qual eles seriam utilizados para Bancos.

In [ ]:
import sqlite3

conn = sqlite3.connect("banco.db")
cursor = conn.cursor()

try:
    cursor.execute("INSERT INTO contas (usuario, saldo) VALUES (?, ?)", ("Alice", 1000))
    cursor.execute("INSERT INTO contas (usuario, saldo) VALUES (?, ?)", ("Bob", 500))

    conn.commit()  # CONFIRMA tudo se não houver erros
    print("Transação concluída com sucesso!")

except sqlite3.Error as e:
    conn.rollback()  # DESFAZ tudo se algo falhar
    print(f"Erro: {e}")

finally:
    conn.close()  # Sempre fecha a conexão

Observações:

- try -> Tentará executar os comandos definidos no seu corpo. Caso não ocorra algum erro o 'conn.commit()' fará a conexão normalmente;

- except -> A 'excessão' é para caso ocorra algum erro e então ele irá descrever o erro e utilizará o 'conn.rollback()'. O método rollback() será explicado daqui a pouco;

- finally -> Essa palavra reservada é fundamental pois ela finaliza o processo de conexão com o banco de dados de forma segura! Independente se o resultado for o 'try' ou o 'except'!

### ***1.2.1 Por que usar `rollback()`?*** 

Imagine uma transferência bancária:

In [ ]:
try:
    # Debita de Alice
    cursor.execute("UPDATE contas SET saldo = saldo - 200 WHERE usuario = 'Alice'")
    
    # Simula um erro no meio da operação
    raise Exception("Falha no sistema!")
    
    # Isso nunca seria executado sem o rollback
    cursor.execute("UPDATE contas SET saldo = saldo + 200 WHERE usuario = 'Bob'")
    
    conn.commit()

except Exception as e:
    conn.rollback()  # Alice não perde os R$200 se Bob não receber
    print(f"Transferência cancelada: {e}")

Sem o `rollback()`, Alice perderia R$200 e Bob não receberia nada — o banco ficaria em estado inválido.

----------------------------------------------------------------------------------------------------

## ***1.3. Controle com `with` (Context Manager)*** 

O Python permite usar `with` diretamente na conexão, que já faz `commit` ou `rollback` automaticamente:

In [ ]:
import sqlite3

conn = sqlite3.connect("banco.db")

with conn:  # commit automático no sucesso, rollback no erro
    conn.execute("INSERT INTO contas (usuario, saldo) VALUES (?, ?)", ("Carlos", 750))

conn.close()

***Atenção***: *o with conn faz commit/rollback automático, mas não fecha a conexão. Você ainda precisa chamar conn.close() depois, ou usar um segundo with.*

Para fechar automaticamente também:

In [ ]:
with sqlite3.connect("banco.db") as conn:
    conn.execute("INSERT INTO contas (usuario, saldo) VALUES (?, ?)", ("Carlos", 750))
# fecha aqui automaticamente

----------------------------------------------------------------------------------------------------

## ***1.4. Transações Explícitas (controle total)*** 

Por padrão, o Python inicia uma transação automaticamente. Mas você pode controlar isso manualmente:

In [ ]:
conn = sqlite3.connect("banco.db")
conn.isolation_level = None  # Modo autocommit (sem transação automática)

cursor = conn.cursor()

cursor.execute("BEGIN")  # Inicia a transação manualmente
try:
    cursor.execute("INSERT INTO produtos (nome, preco) VALUES (?, ?)", ("Notebook", 3500))
    cursor.execute("INSERT INTO estoque (produto_id, quantidade) VALUES (?, ?)", (1, 10))
    cursor.execute("COMMIT")  # Confirma
except:
    cursor.execute("ROLLBACK")  # Desfaz

### ***1.4.1 Tipos de Transação no SQLite*** 

Você pode especificar o nível de isolamento ao iniciar:

In [ ]:
cursor.execute("BEGIN DEFERRED")   # Padrão — adquire lock só quando necessário
cursor.execute("BEGIN IMMEDIATE")  # Adquire lock de escrita imediatamente
cursor.execute("BEGIN EXCLUSIVE")  # Bloqueia o banco inteiro para escrita

| Tipo       | Quando usar                |
| ----------- | -------------------------- |
| `DEFERRED`   | Maioria dos casos (padrão) |
| `IMMEDIATE`  | Quando você sabe que vai escrever e quer evitar conflitos|
| `EXCLUSIVE`      | Operações críticas que não podem ter nenhuma leitura simultânea|

### ***1.4.2 Savepoints (transações aninhadas)*** 

Permitem criar "pontos de restauração" dentro de uma transação:

In [ ]:
conn = sqlite3.connect("banco.db")
cursor = conn.cursor()

try:
    cursor.execute("INSERT INTO pedidos (produto) VALUES ('Teclado')")
    
    cursor.execute("SAVEPOINT ponto1")  # Marca o ponto
    
    try:
        cursor.execute("INSERT INTO pedidos (produto) VALUES ('Mouse')")
        # Simula erro
        raise Exception("Produto indisponível")
        cursor.execute("RELEASE SAVEPOINT ponto1")  # Confirma o savepoint
        
    except Exception as e:
        cursor.execute("ROLLBACK TO SAVEPOINT ponto1")  # Volta só até aqui
        print(f"Revertido ao savepoint: {e}")

    conn.commit()  # O INSERT do Teclado ainda é salvo!

except:
    conn.rollback()

conn.close()

Resultado: o "Teclado" é salvo, o "Mouse" não — sem cancelar a transação inteira.

----------------------------------------------------------------------------------------------------

## ***1.5. Capturando Erros Específicos*** 

O módulo `sqlite3` tem uma hierarquia de exceções:

In [ ]:
try:
    cursor.execute("INSERT INTO usuarios (id, nome) VALUES (1, 'Ana')")
    cursor.execute("INSERT INTO usuarios (id, nome) VALUES (1, 'Bruno')")  # id duplicado!
    conn.commit()

except sqlite3.IntegrityError as e:
    # Violação de chave primária, UNIQUE, NOT NULL, etc.
    conn.rollback()
    print(f"Erro de integridade: {e}")

except sqlite3.OperationalError as e:
    # Tabela não existe, banco bloqueado, SQL inválido, etc.
    conn.rollback()
    print(f"Erro operacional: {e}")

except sqlite3.DatabaseError as e:
    # Qualquer erro de banco genérico
    conn.rollback()
    print(f"Erro no banco: {e}")

except sqlite3.Error as e:
    # Base de todos os erros do sqlite3
    conn.rollback()
    print(f"Erro desconhecido: {e}")

### ***1.5.1 Hierarquia completa:*** 

    sqlite3.Error
    ├── sqlite3.DatabaseError
    │   ├── sqlite3.IntegrityError   # Violação de constraints
    │   ├── sqlite3.OperationalError # Erros de operação
    │   ├── sqlite3.ProgrammingError # Erros de código (SQL errado)
    │   ├── sqlite3.DataError        # Dados inválidos
    │   └── sqlite3.NotSupportedError
    └── sqlite3.Warning

----------------------------------------------------------------------------------------------------

# ***2. Boas Práticas — Resumo*** 

In [ ]:
import sqlite3

def transferir(origem, destino, valor):
    with sqlite3.connect("banco.db") as conn:
        try:
            cursor = conn.cursor()
            
            cursor.execute("UPDATE contas SET saldo = saldo - ? WHERE usuario = ?", (valor, origem))
            cursor.execute("UPDATE contas SET saldo = saldo + ? WHERE usuario = ?", (valor, destino))
            
            conn.commit()
            print("Transferência realizada!")

        except sqlite3.IntegrityError as e:
            conn.rollback()
            print(f"Erro de integridade: {e}")

        except sqlite3.OperationalError as e:
            conn.rollback()
            print(f"Erro operacional: {e}")

transferir("Alice", "Bob", 200)

## ***2.1. Regras de ouro:*** 

1. Sempre use `try/except/finally` em operações críticas;

2. Sempre chame `rollback()` no `except`;

3. Sempre feche a conexão no `finally` ou use `with`;

4. Nunca use f-strings em queries — use `?` como placeholder;

5. Use Savepoints quando precisar de controle granular dentro de uma transação.